# Case Studies in Health and SaaS
## Module 5, Lesson 7

In this notebook, we will:
1. Reproduce the Obermeyer et al. bias pattern using synthetic healthcare data
2. Analyze fairness in a credit scoring model
3. Implement and compare bias mitigation strategies
4. Visualize the impact of different fairness definitions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

np.random.seed(42)

## 1. Reproducing the Obermeyer Bias Pattern

We simulate the situation where cost is used as a proxy for health need, and access to care differs by race.

In [ ]:
n = 5000

# Race (0 = white, 1 = Black)
race = np.random.choice([0, 1], size=n, p=[0.7, 0.3])

# True health need (number of chronic conditions)
# Slightly higher need in Black population due to systemic factors
health_need = np.random.poisson(lam=2.0 + 0.5 * race)

# Access to care (lower for Black patients)
access_to_care = np.where(race == 0,
                          np.random.beta(8, 2, n),  # higher access
                          np.random.beta(4, 6, n))  # lower access

# Healthcare costs: determined by need AND access
# Patients with high need + high access = high cost
# Patients with high need + low access = medium cost (under-treated)
cost = health_need * access_to_care * 5000 + np.random.normal(0, 2000, n)
cost = np.clip(cost, 0, None)

df_health = pd.DataFrame({
    'race': race,
    'health_need': health_need,
    'access_to_care': access_to_care,
    'cost': cost
})

print("Average Cost by Race:")
print(df_health.groupby('race')['cost'].mean().round(0))
print("\nAverage Health Need by Race:")
print(df_health.groupby('race')['health_need'].mean().round(2))
print("\nAverage Access to Care by Race:")
print(df_health.groupby('race')['access_to_care'].mean().round(3))

In [ ]:
# Train model using COST as proxy for need (the biased approach)
X_cost = pd.DataFrame({'cost': df_health['cost']})
y_high_cost = (df_health['cost'] > df_health['cost'].median()).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_cost, y_high_cost, test_size=0.3, random_state=42)
race_test = df_health.iloc[X_test.index]['race']

model_cost = LogisticRegression()
model_cost.fit(X_train, y_train)
y_pred_cost = model_cost.predict(X_test)

print("=== Model Using Cost as Proxy for Need ===")
for r in [0, 1]:
    mask = race_test == r
    actual_need = df_health.iloc[X_test.index].loc[mask, 'health_need'].mean()
    pred_high = y_pred_cost[mask].mean()
    print(f"  Race {r}: Avg Actual Need = {actual_need:.2f}, "
          f"Predicted High-Risk = {pred_high:.3f}")

# Train model using ACTUAL health need (the fair approach)
X_need = pd.DataFrame({'health_need': df_health['health_need']})
y_high_need = (df_health['health_need'] > df_health['health_need'].median()).astype(int)

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_need, y_high_need, test_size=0.3, random_state=42)
race_test2 = df_health.iloc[X_test2.index]['race']

model_need = LogisticRegression()
model_need.fit(X_train2, y_train2)
y_pred_need = model_need.predict(X_test2)

print("\n=== Model Using Actual Health Need ===")
for r in [0, 1]:
    mask = race_test2 == r
    pred_high = y_pred_need[mask].mean()
    print(f"  Race {r}: Predicted High-Risk = {pred_high:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bias from cost proxy
groups = ['White', 'Black']
for i, r in enumerate([0, 1]):
    mask_cost = race_test == r
    mask_need = race_test2 == r
    cost_pred = y_pred_cost[mask_cost].mean()
    need_pred = y_pred_need[mask_need].mean()
    axes[0].bar(i - 0.15, cost_pred, 0.3, label='Cost proxy' if i == 0 else '', color='#DD8452')
    axes[0].bar(i + 0.15, need_pred, 0.3, label='Actual need' if i == 0 else '', color='#4C72B0')

axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(groups)
axes[0].set_ylabel('Proportion Predicted High-Risk')
axes[0].set_title('Obermeyer-Style Bias: Cost Proxy vs. Actual Need')
axes[0].legend()

# True need by group (should reverse the disparity)
true_need_by_race = df_health.groupby('race')['health_need'].mean()
axes[1].bar(groups, true_need_by_race.values, color=['#4C72B0', '#DD8452'])
axes[1].set_ylabel('Average Health Need (Chronic Conditions)')
axes[1].set_title('True Health Need by Race')

plt.tight_layout()
plt.show()
print("Note: When cost is used as proxy, Black patients are less likely to be classified as high-risk")
print("despite having higher actual health need. This reproduces the Obermeyer finding.")

## 2. Credit Scoring Fairness Analysis

We examine a synthetic credit dataset for fairness across demographic groups.

In [ ]:
def generate_credit_data(n=5000):
    """Generate synthetic credit data with demographic attributes."""
    race = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
    gender = np.random.choice([0, 1], size=n, p=[0.5, 0.5])
    
    income = np.where(race == 0,
                      np.random.normal(70, 25, n),
                      np.random.normal(50, 20, n))
    credit_score = np.where(race == 0,
                            np.random.normal(700, 50, n),
                            np.random.normal(650, 60, n))
    loan_amount = np.random.uniform(1000, 50000, n)
    debt_ratio = np.random.uniform(0.1, 0.7, n)
    
    default_prob = 0.05 + 0.4 * (credit_score < 640).astype(float) + \
                   0.2 * (debt_ratio > 0.5).astype(float) + \
                   0.02 * race  # small structural difference
    default = np.random.binomial(1, np.clip(default_prob, 0.02, 0.6))
    
    return pd.DataFrame({
        'race': race,
        'gender': gender,
        'income': income.round(1),
        'credit_score': credit_score.round(0),
        'loan_amount': loan_amount.round(0),
        'debt_ratio': debt_ratio.round(3),
        'default': default
    })

df_credit = generate_credit_data()
print("Credit Data Summary:")
print(df_credit.groupby('race')[['income', 'credit_score', 'default']].mean().round(2))

In [ ]:
def compute_all_fairness_metrics(y_true, y_pred, protected_attr, group_names=None):
    """Compute standard fairness metrics for each group."""
    groups = np.unique(protected_attr)
    results = {}
    
    for g in groups:
        mask = protected_attr == g
        yt = y_true[mask]
        yp = y_pred[mask]
        
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        name = group_names[g] if group_names else f'Group {g}'
        results[name] = {
            'N': len(yt),
            'Base Rate': yt.mean(),
            'Approval Rate': yp.mean(),
            'TPR': tpr,
            'TNR': tnr,
            'FPR': fpr,
            'FNR': fnr,
            'Precision': precision
        }
    
    return results

# Train model
X = df_credit[['income', 'credit_score', 'loan_amount', 'debt_ratio']]
y = df_credit['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

protected = df_credit.iloc[X_test.index]['race']
metrics = compute_all_fairness_metrics(
    y_test.values, y_pred, protected.values,
    group_names={0: 'White', 1: 'Non-White'}
)

metrics_df = pd.DataFrame(metrics).T
print("Fairness Metrics by Group:")
print(metrics_df.round(4))

## 3. Comparsion of Mitigation Strategies

We compare three approaches: (1) no mitigation, (2) sample reweighing, (3) threshold adjustment.

In [ ]:
# Strategy 1: Reweighing
train_df = df_credit.iloc[X_train.index]
weights = np.ones(len(train_df))
for r in [0, 1]:
    mask = train_df['race'] == r
    weights[mask] = len(train_df) / (2 * mask.sum())

model_weighted = GradientBoostingClassifier(n_estimators=100, random_state=42)
model_weighted.fit(X_train, y_train, sample_weight=weights)
y_pred_weighted = model_weighted.predict(X_test)

# Strategy 2: Threshold adjustment to equalize TPR
y_prob = model.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0.1, 0.9, 100)

best_threshold = 0.5
min_tpr_diff = float('inf')
for t in thresholds:
    y_pred_t = (y_prob > t).astype(int)
    tpr_by_group = []
    for r in [0, 1]:
        mask = protected.values == r
        yt = y_test.values[mask]
        yp = y_pred_t[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        tpr_by_group.append(tpr)
    diff = abs(tpr_by_group[0] - tpr_by_group[1])
    if diff < min_tpr_diff:
        min_tpr_diff = diff
        best_threshold = t

y_pred_threshold = (y_prob > best_threshold).astype(int)

# Compare all strategies
strategies = {
    'No Mitigation': y_pred,
    'Reweighing': y_pred_weighted,
    f'Threshold (t={best_threshold:.2f})': y_pred_threshold
}

print(f"Best threshold for equal TPR: {best_threshold:.3f}")
print("\n=== Comparsion of Mitigation Strategies ===")
for name, preds in strategies.items():
    metrics = compute_all_fairness_metrics(
        y_test.values, preds, protected.values,
        group_names={0: 'White', 1: 'Non-White'}
    )
    white_approx = metrics['White']['Approval Rate']
    nonwhite_approx = metrics['Non-White']['Approval Rate']
    tpr_diff = abs(metrics['White']['TPR'] - metrics['Non-White']['TPR'])
    fpr_diff = abs(metrics['White']['FPR'] - metrics['Non-White']['FPR'])
    acc = (preds == y_test.values).mean()
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  White approval rate: {white_approx:.4f}")
    print(f"  Non-White approval rate: {nonwhite_approx:.4f}")
    print(f"  TPR diff: {tpr_diff:.4f}")
    print(f"  FPR diff: {fpr_diff:.4f}")

## 4. Exercises

### Basic
1. In the Obermeyer simulation, why does using cost as a proxy for need produce biased predictions?
2. Compare the approval rates for White and Non-White groups in the credit model. Is there disparate impact?

### Implementation
3. Add gender as a protected attribute to the credit analysis. Are there intersectional effects (race + gender)?
4. Implement equalized odds mitigation: find thresholds that equalize both TPR and FPR simultaneously.

### Critical Thinking
5. The Obermeyer simulation uses a simplified model of the real-world case. What important factors are missing? How might including them change the results?
6. Which mitigation strategy (reweighing or threshold adjustment) is more appropriate for credit scoring? Consider regulatory requirements (ECOA adverse action notice) and explainability.
7. Should a credit scoring model be required to achieve demographic parity even if the groups have different base default rates? Defend your position.